# H SC 790 · Lab 2 · Build a Pipeline That Checks Itself

**Week 3.** Due Thursday, February 11, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Rows balance: in = loaded + quarantined + deduplicated
- Report is readable without the code
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/OWNER/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

Two monthly batches arrive from the same intake system as Lab 1. Build a pipeline that lands each batch raw, stages it, checks it, quarantines what fails, loads the rest, and writes a quality report someone can read without the code. Then run the second batch through the same code, unchanged.

This notebook starts from the reference schema in `lab02/reference_schema.sql`. If you prefer your own Lab 1 schema, paste your DDL in its place.

In [ ]:
import duckdb
con = duckdb.connect()
con.sql(read_text("lab02/reference_schema.sql"))
con.sql(f"INSERT INTO ref_status SELECT * FROM read_csv_auto('{COURSE_URL}lab02/ref_status.csv')")
con.sql(f"INSERT INTO condition SELECT * FROM read_csv_auto('{COURSE_URL}lab02/ref_condition.csv')")
con.sql(f"CREATE TABLE ref_race AS SELECT * FROM read_csv_auto('{COURSE_URL}lab02/ref_race.csv')")
con.sql("SHOW TABLES").df()

## 1. Land the raw batch

Load the batch into a raw table exactly as received, adding only a load timestamp. Never edit it. Write this as a function of the batch file name, because step 5 reuses it.

In [ ]:
def land(batch_file):
    """Append one batch file to raw_batch, as received, with a load timestamp. Returns the number of rows landed."""
    # YOUR CODE HERE
    pass

print(land("batch_1.csv"), "rows landed")

## 2. Stage it

Parse, cast, trim, and standardize into staging tables (`stg_case`, `stg_patient`, and whatever else you need). Count rows in and out at every step. Decide here what you do with exact duplicate rows, and count them.

In [ ]:
def stage(batch_file):
    """Rebuild the staging tables from one landed batch. Returns counts for the accounting identity."""
    # YOUR CODE HERE
    pass

counts = stage("batch_1.csv")
counts

## 3. Write the rule list

At least eight checks: two or more from each Kahn category (completeness, conformance, plausibility), plus two timeliness intervals. Each check is a row: category, name, a SQL query that returns one number, a limit, and a severity (`warn`, `quarantine`, or `block`). The first four are the slide's. Add yours.

Think about what a row-level check cannot see.

In [ ]:
checks = [
 # category,      name,                 SQL returning one number,         limit, severity
 ('completeness','race_missing_pct',
  "SELECT 100.0*COUNT(*) FILTER (WHERE race IS NULL)/COUNT(*) FROM stg_patient", 15, 'warn'),
 ('conformance', 'bad_status_n',
  "SELECT COUNT(*) FROM stg_case WHERE case_status NOT IN (SELECT code FROM ref_status)", 0, 'quarantine'),
 ('plausibility','report_before_specimen_n',
  "SELECT COUNT(*) FROM stg_case WHERE report_date < specimen_date", 0, 'quarantine'),
 ('timeliness',  'median_result_to_report_days',
  "SELECT MEDIAN(report_date - result_date) FROM stg_case", 2, 'warn'),
 # YOUR CODE HERE
]

## 4. Run, quarantine, load

Run the checks into a report table. Move failing rows to quarantine with the reason. Load the rest into the registry. Prove the accounting identity: received = loaded + quarantined + deduplicated.

In [ ]:
con.sql("""CREATE TABLE IF NOT EXISTS quarantine AS
           SELECT *, ''::VARCHAR AS reason, now() AS quarantined_at FROM
           (SELECT 0 AS case_id, 0 AS patient_id, '' AS condition_code, 0 AS facility_id, DATE '2000-01-01' AS onset_date,
                   DATE '2000-01-01' AS specimen_date, DATE '2000-01-01' AS result_date, DATE '2000-01-01' AS report_date,
                   '' AS case_status, '' AS batch_id) WHERE false""")

def run_checks():
    rows = []
    for cat, name, sql, limit, sev in checks:
        got = con.sql(sql).fetchall()
        value = float(got[0][0]) if got and got[0][0] is not None else 0.0
        rows.append((cat, name, round(value, 2), limit, sev, 'PASS' if abs(value) <= limit else 'FAIL'))
    return pd.DataFrame(rows, columns=['category', 'check', 'value', 'limit', 'severity', 'result'])

def quarantine_and_load(counts):
    """Quarantine failing rows with a reason, load the rest, and return the completed accounting."""
    # quarantine failing rows, with the reason (slide)
    con.sql("""
      INSERT INTO quarantine
      SELECT *, 'report_before_specimen' AS reason,
             now() AS quarantined_at
      FROM stg_case
      WHERE report_date < specimen_date
        AND case_id NOT IN (SELECT case_id FROM quarantine)
    """)
    # YOUR CODE HERE
    pass
    # load only rows that are not in quarantine (slide)
    con.sql("""
      INSERT OR IGNORE INTO case_report
      SELECT s.* EXCLUDE (batch_id) FROM stg_case s
      ANTI JOIN quarantine q USING (case_id)
    """)
    # YOUR CODE HERE
    pass

report_1 = run_checks()
acct_1 = quarantine_and_load(counts)
# the accounting identity must balance
assert acct_1["received"] == acct_1["loaded"] + acct_1["quarantined"] + acct_1["deduplicated"] + acct_1["already_loaded"]
print(acct_1)
report_1

## 5. Run the second batch

Through the same code, without changing it. Compare the two quality reports. Then look behind each failing check: which facility or laboratory is responsible?

In [ ]:
def run_batch(batch_file):
    land(batch_file)
    counts = stage(batch_file)
    report = run_checks()
    acct = quarantine_and_load(counts)
    assert acct["received"] == acct["loaded"] + acct["quarantined"] + acct["deduplicated"] + acct["already_loaded"]
    return report, acct

report_2, acct_2 = run_batch("batch_2.csv")
print(acct_2)
report_1.merge(report_2, on=["category", "check", "limit", "severity"], suffixes=("_batch1", "_batch2"))

In [ ]:
# YOUR CODE HERE

## 6. Prove idempotency

Run the second batch again. Show that the curated row counts do not change.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. Which check failed most, and for which source?** Give the value, the limit, and the facility or laboratory.

*Your answer here.*

**2. What changed between batch 1 and batch 2?** List every difference your report detected, using only the report.

*Your answer here.*

**3. Which threshold would you change, and why?** Name the use the data serves and the baseline you used.

*Your answer here.*

**4. Who should receive each failing check?** For two failures, name the role that can fix it and what they would do.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab02/` of your course repository.